# Multi-Agent E-commerce Dispute Resolution — Kaggle runner

Notebook này là entrypoint mỏng cho package `multiagent_a2a`. Business logic, agents, verifier, Qwen gateway, trace và QA nằm trong `src/`.

Model contract: chỉ dùng Qwen3-8B đã attach/cache cục bộ; notebook không cài dependency, không gọi Hub và không tải model. Khi model không sẵn sàng, deterministic fallback vẫn tạo đủ 50 output.

In [ ]:
import os


def env_flag(name, default):
    return os.getenv(name, '1' if default else '0').strip().lower() in {'1', 'true', 'yes', 'on'}


# Kaggle overrides. Keep None to use safe auto-discovery.
DATA_DIR = None       # Example: /kaggle/input/my-project/data
INPUT_DIR = None      # Example: /kaggle/input/my-project/input
MODEL_PATH = None     # Example: /kaggle/input/qwen3-8b/transformers/default/1
WORK_ROOT = None      # Defaults to /kaggle/working on Kaggle

ENABLE_LLM = env_flag('EC_ENABLE_LLM', True)     # Local/attached model only; never downloads
FORCE_RULE_FALLBACK = env_flag('EC_FORCE_RULE_FALLBACK', False)
STRICT_OFFICIAL_ASSERTIONS = env_flag('EC_STRICT_OFFICIAL_ASSERTIONS', True)
MIRROR_LOGGING = env_flag('EC_MIRROR_LOGGING', True)


In [ ]:
from pathlib import Path
import json
import sys


def find_project_root():
    preferred = [Path.cwd(), Path.cwd().parent]
    for candidate in preferred:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'multiagent_a2a').is_dir():
            return candidate.resolve()
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob('pyproject.toml'), key=lambda path: (len(path.parts), str(path)))
        for marker in matches:
            if (marker.parent / 'src' / 'multiagent_a2a').is_dir():
                return marker.parent.resolve()
    raise FileNotFoundError('Cannot find project source. Attach the full repository, including pyproject.toml and src/.')


PROJECT_ROOT = find_project_root()
source_dir = str(PROJECT_ROOT / 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

from multiagent_a2a import RunConfig, run_pipeline

config = RunConfig.from_env(search_root=PROJECT_ROOT)
overrides = {
    'enable_llm': ENABLE_LLM and not FORCE_RULE_FALLBACK,
    'strict_official_assertions': STRICT_OFFICIAL_ASSERTIONS,
    'mirror_logging': MIRROR_LOGGING,
}
if DATA_DIR is not None:
    overrides['data_dir'] = DATA_DIR
if INPUT_DIR is not None:
    overrides['input_dir'] = INPUT_DIR
if MODEL_PATH is not None:
    overrides['model_path'] = MODEL_PATH
if WORK_ROOT is not None:
    overrides['work_root'] = WORK_ROOT
config = config.with_overrides(**overrides)
print(json.dumps(config.describe(), ensure_ascii=False, indent=2))


In [ ]:
report = run_pipeline(
    config,
    progress_callback=lambda done, total: print(f'Processed {done}/{total} cases'),
)
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=2))


In [ ]:
# Inspect metadata to confirm whether Qwen was used or fallback was selected.
metadata = json.loads(config.metadata_path.read_text(encoding='utf-8'))
print(json.dumps(metadata['model'], ensure_ascii=False, indent=2))

try:
    from IPython.display import FileLink, display
    display(FileLink(str(config.submission_zip)))
except ImportError:
    print(config.submission_zip)


## Kaggle checklist

1. Attach full repo, Olist/case data và Qwen3-8B model asset.
2. Enable GPU. Chỉ điền path overrides nếu auto-discovery không chọn đúng asset.
3. Run All; tải `/kaggle/working/submission.zip`.
4. Kiểm tra `metadata.json`: `qwen_validated_cases` cho biết model thực sự tham gia bao nhiêu case; fallback luôn được ghi rõ.